# EDA — base `bank-marketing`

Análise exploratória e tratamento de dados da Fase 1. Cobre as Etapas 1 e 2 do enunciado.

Este notebook **apresenta**; ele não implementa. Toda agregação, todo critério e todo gráfico
vivem em `src/eda.py` e `src/data.py`, e é de lá que a API e o treino da Fase 2 vão consumir as
mesmas funções. Se algo aqui parecer lógica de negócio, está no lugar errado.

**O que esta análise precisa decidir**, e que nenhuma fase posterior consegue decidir sozinha:

1. Qual é o **espaço de braços** definitivo, escolhido pelo suporte amostral observado
2. O que fazer com `euribor3m` e `nr.employed`
3. Qual **split** treino/teste usar
4. Qual **baseline** o bandit precisa superar

Os critérios de cada decisão foram fixados **antes** de rodar a análise — estão escritos na
seção correspondente, junto do número que os resolveu.

## 1. Setup e reprodutibilidade

O kernel do Jupyter roda com o diretório do notebook como raiz, então a raiz do repositório
precisa entrar no path para que `src` seja importável.

In [ ]:
import sys

sys.path.insert(0, "..")

In [ ]:
from src import config, data, eda

eda.set_style()
config.SEED, config.RAW_SHA256[:8], config.RAW_SEPARATOR

In [ ]:
raw = data.load_raw()
raw.shape

O separador é `;`, não `,`. Lido com a vírgula, o pandas devolve um DataFrame de **uma coluna
só, sem levantar erro** — por isso `load_raw` valida o schema depois de ler, em vez de confiar
no parse.

## 2. Perfil geral e cardinalidade

In [ ]:
eda.cardinality_report(raw)

21 colunas, 41.188 linhas, **nenhum nulo**. Três observações que mudam o tratamento:

- `nr.employed` tem 11 valores distintos em 41 mil linhas, e `emp.var.rate` tem 10. Não são
  medidas do cliente — são carimbos de período. A seção 8 fecha isso com número.
- `pdays` tem 27 valores, mas 96,3% deles são o mesmo: `999`, sentinela de "nunca contatado
  antes". Tratar como distância faria o modelo ler `999` como "contato muito antigo".
- `duration` tem 1.544 valores e é a coluna proibida. Seção 5.

In [ ]:
raw["pdays"].eq(config.PDAYS_SENTINEL).mean().round(4)

In [ ]:
eda.conversion_by(data.prepare(raw), config.FIRST_CONTACT_COLUMN)

A sentinela separa dois mundos: quem nunca foi contatado converte a **9,26%**, quem já tinha
histórico converte a **63,83%**. Os intervalos não chegam perto de se tocar. Por isso `prepare`
deriva `first_contact` como flag explícita em vez de deixar o `999` solto na coluna numérica.

## 3. `unknown` é resposta, não ausência

In [ ]:
eda.unknown_report(raw)

`unknown` aparece em 6 colunas, com peso muito desigual: 20,9% em `default` contra 0,2% em
`marital`.

**Decisão: `unknown` não é imputado.** No `bank-marketing` ele registra que a informação não foi
obtida na ligação, o que é em si um sinal — e o `HistGradientBoostingClassifier` da Fase 2 trata
categoria nativamente. Imputar aqui seria inventar dado, o que o enunciado veda. `prepare`
preserva o nível, e a limitação fica documentada no README.

## 4. Conversão global e por segmento

In [ ]:
df = data.prepare(raw)
df.shape

In [ ]:
eda.conversion_by(df.assign(total="base inteira"), "total")

Taxa-base de **11,27%** (4.640 de 41.188), IC95% de Wilson `[10,96%; 11,57%]`. É contra este
número que todo o resto se compara.

Usamos Wilson e não a aproximação normal porque as células de braço mais magras têm poucas
conversões, e ali a normal produz intervalo que atravessa o zero.

In [ ]:
eda.conversion_by(df, "poutcome")

In [ ]:
eda.conversion_by(df, "job")

Heterogeneidade forte no contexto: `poutcome = success` converte a 65,1% contra 8,8% de
`nonexistent`; por ocupação, `student` (31,4%) e `retired` (25,2%) contra `blue-collar` (6,9%).

Isso é boa notícia para a formulação: **o contexto discrimina**. A pergunta que a seção 6
responde é outra e mais difícil — se a *ação* também discrimina.

## 5. `duration`: o vazamento em números

In [ ]:
eda.duration_leakage(raw)

O enunciado proíbe `duration` nominalmente. Vale mostrar por quê com número em vez de citação:

| conjunto | AUC |
|---|---|
| 19 features legítimas + `duration` | **0,955** |
| 19 features legítimas | 0,816 |
| **só `duration`** | **0,819** |

Uma única coluna supera sozinha todas as features legítimas juntas. Isso não é poder preditivo,
é vazamento: a duração da ligação só existe **depois** que a ligação terminou, e ligação que
termina em venda dura mais. No momento em que a política precisa decidir a quem ligar, esse
número não existe.

`config.FORBIDDEN_COLUMNS` remove a coluna, e `prepare` é o único caminho de entrada do resto do
projeto — a API e o ambiente da Fase 2 nunca a veem.

In [ ]:
fig = eda.plot_duration_leakage(raw)
eda.save_figure(fig, 'duration_vazamento.png')
fig

## 6. O espaço de braços — a decisão central

Um bandit precisa escolher entre ações. O log tem uma ação só ("ligamos e oferecemos depósito"),
mas registra **como** ela foi executada: `contact` (canal), `month` e `day_of_week`. É daí que
saem os braços — e a consequência é que todo braço tem suporte real no log, sem recompensa
sintética.

`month` fica **fora** do braço. Cruzá-lo daria 2 × 10 × 5 = 100 células, e ele é o principal
confundidor temporal (seção 7). `day_of_week` entra agregado em três janelas, conforme a
mitigação prevista no plano.

**Critério pré-registrado:** o pior braço do espaço precisa de **≥ 1.000 eventos e ≥ 100
conversões**. Mil eventos dão erro-padrão de ~1 p.p. sobre uma taxa de 11%; cem conversões é o
piso para a taxa não oscilar com um punhado de casos.

In [ ]:
eda.conversion_by(df, "day_of_week")

In [ ]:
eda.conversion_by(df, config.WEEK_WINDOW_COLUMN)

A agregação não é arbitrária: segunda é o dia distintamente pior (9,95%), terça/quarta/quinta
formam um bloco apertado (11,67%–12,12%) e sexta fica no meio (10,81%). As janelas
`early` / `mid` / `late` capturam exatamente essa estrutura.

In [ ]:
eda.screen_arm_spaces(df)

**Os três candidatos passam no piso.** O critério pré-registrado não discriminou — então o
desempate abaixo foi aplicado *depois* de ver o dado, e vale registrar isso honestamente.

Desempate, por dois argumentos:

1. **Granularidade que não distingue não serve.** Dentro de `cellular`, os intervalos de
   terça (`[14,8%; 16,8%]`), quarta (`[14,3%; 16,3%]`) e quinta (`[14,5%; 16,3%]`) se sobrepõem
   quase inteiramente. Separar `mid` em três braços cria três braços que o bandit não consegue
   diferenciar — ele gastaria exploração para aprender que são iguais.
2. **Custo no replay.** O track C aceita o evento só quando o braço escolhido coincide com o
   registrado, então o aproveitamento cai com `1/K`: 16,7% com 6 braços contra 10% com 10.
   Perder 40% dos eventos aceitos para ganhar braços indistinguíveis é troca ruim.

`contact` sozinho (2 braços) maximizaria o replay, mas apaga a dimensão "quando abordar" — que
é metade do que o enunciado chama de "próximo passo".

**Decisão: `contact` × `week_window`, 6 braços.** O pior deles tem 2.979 eventos e 139
conversões, folgado acima do piso.

In [ ]:
eda.arm_support(df, list(config.ARM_COLUMNS))

In [ ]:
tabela = eda.conversion_by(df, config.ARM_COLUMN)
fig = eda.plot_conversion(tabela, title="Conversão por braço")
eda.save_figure(fig, "cvr_por_braco.png")
fig

### O baseline, e um alerta

O canal domina: `cellular` converte a 14,74% contra 5,23% de `telephone`. Dentro do celular, a
janela move pouco — de 12,79% (`early`) a 15,47% (`mid`).

In [ ]:
df[config.ARM_COLUMN].value_counts(normalize=True)

**Aqui está a armadilha que o plano antecipou, e ela disparou.** O braço mais usado do log
(`cellular|mid`, 38,8% do volume) é *também* o de maior conversão (15,47%). Modal e melhor
histórico são o mesmo braço.

A consequência é direta: se o baseline fosse "melhor braço histórico", qualquer bandit
não-contextual convergiria exatamente para ele e o ganho seria **zero** — falhando o requisito
explícito da Etapa 3.

**Decisão de baseline:** a regra fixa é a **política de log** — o mix que a operação de fato
praticava, cuja conversão realizada é os 11,27% da seção 4. Concentrar em `cellular|mid` rende
15,47%, um uplift de **+37%**, e é um ganho legítimo: a operação não jogava o melhor braço, ela
jogava uma mistura. `BestHistoricalArm` entra como comparador secundário, mais duro, e é contra
ele que a política **contextual** precisa provar valor.

E há um alerta a registrar para a Fase 2: nas estratificações testadas (`job`, `education`,
`marital`, `poutcome`), quando o melhor braço muda de identidade, os intervalos de Wilson dos
concorrentes **se sobrepõem** — nenhuma troca é estatisticamente distinguível. A heterogeneidade
braço × contexto é fraca neste espaço de braços. A Fase 2 tem esse teste no seu checklist e
precisa levá-lo a sério.

## 7. Confounding temporal

In [ ]:
eda.month_run_count(raw)

26 blocos de meses consecutivos em 41 mil linhas — se o arquivo estivesse embaralhado seriam
dezenas de milhares. **A ordem do arquivo é cronológica**, o que torna o índice de período
utilizável mesmo sem coluna de data.

In [ ]:
eda.contact_by_month(df).round(3)

In [ ]:
fig = eda.plot_contact_over_time(df)
eda.save_figure(fig, "contact_por_mes.png")
fig

O mix de canal **não é estável no tempo**: os dois primeiros períodos (mai/jun de 2008) são
100% `telephone`, e a partir de agosto o celular passa de 97%. O banco migrou de canal ao longo
da campanha.

Isso significa que parte da vantagem medida do `cellular` **não é do canal** — é da época em que
ele foi usado. Não temos como separar as duas coisas com este log: seria preciso ter contatado
o mesmo perfil pelos dois canais no mesmo período, e a operação não fez isso.

Não é problema que se resolva, é limitação que se declara. Vai para o README e é exatamente o
tipo de viés que o track C (replay sobre o log real) existe para contrabalançar.

## 8. Os indicadores macro são carimbo de calendário

In [ ]:
eda.macro_calendar_report(df)

In [ ]:
fig = eda.plot_macro_over_time(df)
eda.save_figure(fig, "macro_no_tempo.png")
fig

**Critério pré-registrado:** R² do indicador contra o índice de período ≥ 0,95 significa proxy
de calendário.

O resultado é mais extremo do que o critério pedia. `emp.var.rate`, `cons.price.idx`,
`cons.conf.idx` e `nr.employed` têm **R² = 1,0000** — são literalmente constantes dentro de cada
período. `euribor3m` fica em 0,9996. E para `cons.price.idx`, cada valor identifica **um único
período**: saber o índice é saber a data.

Um indicador constante dentro do período **não personaliza nada**. Ele move a taxa-base, não
distingue um cliente do outro. Deixá-lo no contexto faz o modelo acertar pelo momento econômico
e parecer que acertou pelo perfil.

**Decisão:** as cinco colunas ficam na base — são conjuntura real e o banco as conhece no
momento da decisão — mas passam a viver em `config.MACRO_COLUMNS`, separadas de
`config.CLIENT_COLUMNS`. O ambiente da Fase 2 usa o contexto de cliente por padrão, e roda a
versão com macro como ablação. Assim a pergunta "quanto do acerto vem do calendário" deixa de
ser retórica e vira medida.

Destemporalizar (usar variação em vez de nível) foi descartado: sem coluna de data, a variação
só seria computável sobre a ordem de linha — tão temporal quanto o nível, e menos interpretável.

## 9. Preparação e split

In [ ]:
train, test = data.split_train_test(df)
train.shape, test.shape

**Critério pré-registrado:** o split temporal é rejeitado como principal se algum braço ficar
sem suporte no fold de teste, ou se a diferença de conversão entre treino e teste passar de
5 p.p.

In [ ]:
train[config.TARGET_BINARY].mean(), test[config.TARGET_BINARY].mean()

In [ ]:
eda.arm_support(test, list(config.ARM_COLUMNS))

Estratificado: conversão de 11,263% no treino contra 11,277% no teste — diferença de 0,014 p.p.
Os seis braços aparecem no fold de teste, o mais magro com 596 eventos e 28 conversões.

O split temporal (últimos 20% do arquivo) dá conversão de **6,37% no treino contra 30,83% no
teste** — 24,5 p.p. de diferença, quase cinco vezes o limite do critério. A causa é a mesma da
seção 7: a campanha mudou de canal, de público e de conjuntura ao longo de 2008–2010, e a cauda
do arquivo é uma operação diferente da cabeça.

**Decisão: estratificado como principal**, estratificando por alvo **e** braço — o ambiente da
Fase 2 estima `P(y | contexto, braço)` para cada braço e precisa dos dois lados povoados. O
split temporal fica como análise de sensibilidade na Fase 3, com a ressalva do suporte.

---

## Fechamento: o que esta fase decidiu

| Decisão | Resultado | O número que decidiu |
|---|---|---|
| **Espaço de braços** | `contact` × `week_window` — 6 braços | pior braço: 2.979 eventos / 139 conversões, acima do piso de 1.000 / 100 |
| **Baseline** | Regra fixa = política de log (11,27%) | braço modal e melhor braço coincidem (`cellular|mid`, 38,8% do volume, 15,47% de conversão) |
| **`euribor3m` / `nr.employed`** | Separados em `MACRO_COLUMNS`, fora do contexto padrão | R² contra o período = 1,0000 (0,9996 para `euribor3m`) |
| **Split** | Estratificado por alvo × braço, seed 42 | temporal dá 24,5 p.p. de deriva de conversão contra 0,014 p.p. do estratificado |
| **`unknown`** | Preservado como nível | 20,9% em `default`; imputar inventaria dado |
| **`duration`** | Descartada | sozinha faz AUC 0,819, acima das 19 features legítimas juntas (0,816) |

**Alertas para a Fase 2:** a heterogeneidade braço × contexto é fraca — nenhuma troca de melhor
braço entre estratos é estatisticamente distinguível. E o confounding `contact` × período é
severo (os dois primeiros períodos são 100% `telephone`), então parte do efeito de canal é
época.